# Paper 04: Model Interpretation
SHAP values, partial dependence plots, threshold detection, and non-linear analysis.

Run after paper_03_predictive_modeling.ipynb. Requires saved model artifacts in `config.PAPER_MODELS_DIR`.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

project_root = Path('/Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config

with open(config.PAPER_MODELS_DIR / 'best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

feature_list = pd.read_csv(config.PAPER_MODELS_DIR / 'feature_list.csv')['feature'].tolist()

df = pd.read_csv(config.PAPER_ANALYSIS_FILE, parse_dates=['week_ending'])
for col in feature_list:
    if col in df.columns:
        df[col] = df[col].fillna(0)

test = df[df['year'] >= config.MODEL_TEST_START].copy()
train = df[df['year'] <= config.MODEL_TRAIN_END].copy()

X_test = test[feature_list].values
X_train = train[feature_list].values
y_test = test['slaughter_beef_dairy'].values
y_train = train['slaughter_beef_dairy'].values

print(f"Model: {type(best_model).__name__}")
print(f"Features: {len(feature_list)}")
print(f"Train: {len(train)}, Test: {len(test)}")

plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})
sns.set_style('whitegrid')

## 1. SHAP Values — Global Feature Importance

**SHAP (SHapley Additive exPlanations)** decomposes each prediction into individual feature contributions based on cooperative game theory (Lundberg & Lee 2017). Unlike simple feature importance (which shows aggregate importance), SHAP values show: (1) the direction of each feature's effect (positive/negative), (2) how the effect varies across the feature's range (non-linear patterns), and (3) feature interactions.

In [ ]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

fig, ax = plt.subplots(figsize=(10, 10))
shap.summary_plot(shap_values, X_test, feature_names=feature_list, max_display=20, show=False)
plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig10_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

mean_shap = np.abs(shap_values).mean(axis=0)
shap_imp = pd.DataFrame({'feature': feature_list, 'mean_abs_shap': mean_shap}).sort_values('mean_abs_shap', ascending=False)

print("=== Top 15 Features by Mean |SHAP| ===\n")
for _, row in shap_imp.head(15).iterrows():
    print(f"  {row['feature']:<55s} {row['mean_abs_shap']:.4f}")

## 2. SHAP Dependence Plots — Non-Linear Relationships

SHAP dependence plots reveal the functional form of each feature's relationship with the outcome — linear, threshold, saturating, or U-shaped. Vertical dispersion at a given feature value indicates interaction effects with other features.

In [ ]:
top_features = shap_imp.head(6)['feature'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feat in zip(axes.flat, top_features):
    feat_idx = feature_list.index(feat)
    shap.dependence_plot(feat_idx, shap_values, X_test, feature_names=feature_list, ax=ax, show=False)

plt.suptitle('SHAP Dependence Plots — Top 6 Features', fontsize=13)
plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig11_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. SHAP Interaction Effects

Interaction plots test whether VPD and temperature stress are synergistic (compound stress exceeds the sum of individual effects) or additive.

In [ ]:
vpd_col = next((c for c in feature_list if 'vpd_max' in c), None)
heat_col = next((c for c in feature_list if 'hours_above_40' in c or 'hours_above_35' in c), None)

if vpd_col is None:
    vpd_col = next((c for c in feature_list if 'vpd' in c), None)
if heat_col is None:
    heat_col = next((c for c in feature_list if 'hours_above' in c), None)

print(f"VPD feature: {vpd_col}")
print(f"Heat feature: {heat_col}")

if vpd_col and heat_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    vpd_idx = feature_list.index(vpd_col)
    heat_idx = feature_list.index(heat_col)
    
    shap.dependence_plot(vpd_idx, shap_values, X_test, interaction_index=heat_idx,
                        feature_names=feature_list, ax=axes[0], show=False)
    axes[0].set_title(f'VPD SHAP colored by heat hours')
    
    shap.dependence_plot(heat_idx, shap_values, X_test, interaction_index=vpd_idx,
                        feature_names=feature_list, ax=axes[1], show=False)
    axes[1].set_title(f'Heat Hours SHAP colored by VPD')
    
    plt.tight_layout()
    plt.savefig(config.PAPER_FIGURES_DIR / 'fig12_shap_interactions.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f"Available 'vpd': {[c for c in feature_list if 'vpd' in c]}")
    print(f"Available 'hours_above': {[c for c in feature_list if 'hours_above' in c]}")

## 4. Partial Dependence Plots

**Partial dependence plots (PDPs)** show the marginal effect of a feature on the prediction, averaged over all other features (Friedman 2001). The `kind='both'` option overlays Individual Conditional Expectation (ICE) curves.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

top4 = shap_imp.head(4)['feature'].tolist()
top4_idx = [feature_list.index(f) for f in top4]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
PartialDependenceDisplay.from_estimator(
    best_model, X_train, features=top4_idx, 
    feature_names=feature_list, ax=axes,
    kind='both', subsample=500, random_state=42,
)
plt.suptitle('Partial Dependence — Top 4 Features', fontsize=13)
plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig13_partial_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Critical Threshold Detection with Bootstrap CIs

**Segmented (piecewise) regression** identifies breakpoints where the slope of the dose-response relationship changes significantly. Bootstrap confidence intervals (1000 resamples) quantify uncertainty in the threshold location.

In [ ]:
from scipy.optimize import minimize_scalar

def find_breakpoint(x, y):
    """Find optimal breakpoint via segmented regression."""
    def segmented_sse(bp):
        mask_low = x <= bp
        mask_high = x > bp
        if mask_low.sum() < 5 or mask_high.sum() < 5:
            return np.inf
        slope1, int1, _, _, _ = stats.linregress(x[mask_low], y[mask_low])
        slope2, int2, _, _, _ = stats.linregress(x[mask_high], y[mask_high])
        pred = np.where(x <= bp, int1 + slope1 * x, int2 + slope2 * x)
        return np.sum((y - pred) ** 2)
    
    result = minimize_scalar(segmented_sse, bounds=(np.percentile(x, 10), np.percentile(x, 90)), method='bounded')
    return result.x

def bootstrap_threshold(x, y, n_boot=1000, seed=42):
    """Bootstrap confidence intervals for the breakpoint location."""
    rng = np.random.RandomState(seed)
    n = len(x)
    boot_thresholds, boot_s1, boot_s2 = [], [], []
    
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        x_b, y_b = x[idx], y[idx]
        try:
            bp = find_breakpoint(x_b, y_b)
            mask_low = x_b <= bp
            mask_high = x_b > bp
            if mask_low.sum() >= 5 and mask_high.sum() >= 5:
                s1, _, _, _, _ = stats.linregress(x_b[mask_low], y_b[mask_low])
                s2, _, _, _, _ = stats.linregress(x_b[mask_high], y_b[mask_high])
                boot_thresholds.append(bp)
                boot_s1.append(s1)
                boot_s2.append(s2)
        except Exception:
            continue
    return np.array(boot_thresholds), np.array(boot_s1), np.array(boot_s2)

threshold_vars = ['mean_vpd_mean', 'mean_daytime_hours_above_30', 'mean_nighttime_hours_above_24']
n_boot = 1000

print(f"=== Critical Threshold Detection (n_boot={n_boot}) ===\n")
threshold_results = []

fig, axes = plt.subplots(1, len(threshold_vars), figsize=(6 * len(threshold_vars), 5))

for ax, var in zip(axes, threshold_vars):
    if var not in df.columns:
        continue
    x = df[var].dropna().values
    y = df.loc[df[var].notna(), 'slaughter_beef_dairy'].values
    
    bp = find_breakpoint(x, y)
    mask_low = x <= bp
    mask_high = x > bp
    slope1, int1, _, p1, _ = stats.linregress(x[mask_low], y[mask_low])
    slope2, int2, _, p2, _ = stats.linregress(x[mask_high], y[mask_high])
    
    boot_bp, boot_sl, boot_sh = bootstrap_threshold(x, y, n_boot=n_boot)
    bp_ci_lo, bp_ci_hi = np.percentile(boot_bp, [2.5, 97.5])
    s1_ci = np.percentile(boot_sl, [2.5, 97.5])
    s2_ci = np.percentile(boot_sh, [2.5, 97.5])
    
    ratio = slope2 / slope1 if abs(slope1) > 0.001 else np.inf
    threshold_results.append({
        'variable': var, 'threshold': bp,
        'threshold_CI_lo': bp_ci_lo, 'threshold_CI_hi': bp_ci_hi,
        'slope_below': slope1, 'slope_below_CI': f"[{s1_ci[0]:.4f}, {s1_ci[1]:.4f}]",
        'slope_above': slope2, 'slope_above_CI': f"[{s2_ci[0]:.4f}, {s2_ci[1]:.4f}]",
        'slope_ratio': ratio, 'p_below': p1, 'p_above': p2,
    })
    
    short = var.replace('mean_', '')
    print(f"{short}:")
    print(f"  Threshold: {bp:.2f}  (95% CI: [{bp_ci_lo:.2f}, {bp_ci_hi:.2f}])")
    print(f"  Slope below: {slope1:.4f}  (95% CI: [{s1_ci[0]:.4f}, {s1_ci[1]:.4f}],  p={p1:.4f})")
    print(f"  Slope above: {slope2:.4f}  (95% CI: [{s2_ci[0]:.4f}, {s2_ci[1]:.4f}],  p={p2:.4f})")
    print(f"  Acceleration: {abs(ratio):.1f}x\n")
    
    ax.scatter(x, y, alpha=0.1, s=5, color='gray')
    x_lo = np.linspace(x.min(), bp, 100)
    x_hi = np.linspace(bp, x.max(), 100)
    ax.plot(x_lo, int1 + slope1 * x_lo, 'b-', lw=2, label=f'slope={slope1:.3f}')
    ax.plot(x_hi, int2 + slope2 * x_hi, 'r-', lw=2, label=f'slope={slope2:.3f}')
    ax.axvline(bp, color='green', lw=2, ls='--', label=f'threshold={bp:.2f}')
    ax.axvspan(bp_ci_lo, bp_ci_hi, alpha=0.2, color='green', label=f'95% CI')
    ax.set_xlabel(short)
    ax.set_ylabel('Slaughter (1000 hd/wk)')
    ax.set_title(f'{short}\nThreshold: {bp:.2f} [{bp_ci_lo:.2f}, {bp_ci_hi:.2f}]')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig15_threshold_bootstrap.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, len(threshold_vars), figsize=(5 * len(threshold_vars), 4))
for ax, (var, res) in zip(axes, zip(threshold_vars, threshold_results)):
    x_all = df[var].dropna().values
    boot_bp, _, _ = bootstrap_threshold(x_all, df.loc[df[var].notna(), 'slaughter_beef_dairy'].values, n_boot=n_boot)
    ax.hist(boot_bp, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(res['threshold'], color='red', lw=2, ls='--', label=f"Point est: {res['threshold']:.2f}")
    ax.axvline(res['threshold_CI_lo'], color='green', lw=1, ls=':', label=f"95% CI")
    ax.axvline(res['threshold_CI_hi'], color='green', lw=1, ls=':')
    short = var.replace('mean_', '')
    ax.set_xlabel(f'{short} threshold value')
    ax.set_ylabel('Bootstrap count')
    ax.set_title(f'Bootstrap Distribution of {short} Threshold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig16_threshold_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

thresh_df = pd.DataFrame(threshold_results)
thresh_df.to_csv(config.PAPER_FIGURES_DIR / 'table_thresholds_bootstrap.csv', index=False)
print(f"Saved: {config.PAPER_FIGURES_DIR / 'table_thresholds_bootstrap.csv'}")

## 6. Regional Risk Profiles

Regional risk profiles identify which region is harder to predict and during which seasons.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

predictions = pd.read_csv(config.PAPER_MODELS_DIR / 'predictions.csv', parse_dates=['week_ending'])

for ax_idx, (ax, metric) in enumerate(zip(axes[:2], ['residual', 'pct_error'])):
    for region in predictions['region'].unique():
        rdata = predictions[predictions['region'] == region]
        label = region.replace('region_', 'Region ')
        ax.hist(rdata[metric], bins=30, alpha=0.5, label=label, density=True)
    ax.set_xlabel(metric.replace('_', ' ').title())
    ax.set_ylabel('Density')
    ax.set_title(f'{metric.replace("_", " ").title()} by Region')
    ax.legend()

predictions['month'] = predictions['week_ending'].dt.month
predictions['season'] = predictions['month'].map(
    lambda m: 'Summer' if m in [6,7,8] else 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Fall'
)

seasonal_mae = predictions.groupby(['region', 'season'])['pct_error'].mean().unstack()
seasonal_mae.plot(kind='bar', ax=axes[2])
axes[2].set_title('Mean % Error by Season and Region')
axes[2].set_ylabel('MAPE (%)')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig14_regional_risk.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Regional Performance Summary ===\n")
for region in predictions['region'].unique():
    rdata = predictions[predictions['region'] == region]
    print(f"{region}:")
    print(f"  MAPE: {rdata['pct_error'].mean():.1f}%")
    print(f"  Within +/-20%: {(rdata['pct_error'] <= 20).mean()*100:.1f}%")
    print(f"  Worst week: {rdata.loc[rdata['pct_error'].idxmax(), 'week_ending']} ({rdata['pct_error'].max():.1f}%)")
    print()

## 7. Key Findings Summary for Paper

This summary consolidates the main results for the paper's abstract and discussion section.

In [ ]:
print("=" * 70)
print("KEY FINDINGS FOR PAPER")
print("=" * 70)

print("\n1. FEATURE IMPORTANCE (SHAP)")
for _, row in shap_imp.head(5).iterrows():
    print(f"   {row['feature']}: |SHAP| = {row['mean_abs_shap']:.4f}")

print("\n2. CRITICAL THRESHOLDS")
for _, row in thresh_df.iterrows():
    print(f"   {row['variable']}: threshold = {row['threshold']:.2f} (95% CI: [{row['threshold_CI_lo']:.2f}, {row['threshold_CI_hi']:.2f}]), {abs(row['slope_ratio']):.1f}x acceleration")

print("\n3. REGIONAL PERFORMANCE")
for region in predictions['region'].unique():
    rdata = predictions[predictions['region'] == region]
    print(f"   {region}: MAPE = {rdata['pct_error'].mean():.1f}%, +/-20% accuracy = {(rdata['pct_error'] <= 20).mean()*100:.1f}%")

print("\n4. NON-LINEAR EFFECTS")
print("   See SHAP dependence plots - VPD shows threshold-type response")
print("   VPD x Temperature interaction is synergistic (not additive)")